In [23]:
%%capture --no-stderr

%pip install --quiet -U \
    llama_index \
    faiss_cpu \
    sentence-transformers \
    pymupdf \
    mistralai \
    textwrap3 \
    python-dotenv \
    llama-index-embeddings-huggingface \
    llama-index-vector-stores-faiss \
    llama-index llama-index-readers-web \
    llama-index-llms-mistralai \
    llama-index-llms-openai \
    docling \
    llama-index-llms-llama-api









Python(30620) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [2]:
import os
import time
import textwrap
from typing import Generator
from dotenv import load_dotenv


import faiss
from sentence_transformers import CrossEncoder
from llama_index.core import (
    VectorStoreIndex,
    StorageContext,
    load_index_from_storage,
    PromptTemplate,
    Document
)
from llama_index.core.schema import TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.readers.file import PyMuPDFReader
from llama_index.readers.web import SimpleWebPageReader
from llama_index.llms.mistralai import MistralAI
from llama_index.core import PromptTemplate
from llama_index.llms.openai import OpenAI

import json
import gc
import pickle


from llama_index.core.node_parser import (
    SentenceSplitter,
    SemanticSplitterNodeParser,
)
from docling.document_converter import DocumentConverter
from llama_index.core.indices.query.query_transform import HyDEQueryTransform
from llama_index.core.query_engine import TransformQueryEngine
from llama_index.llms.openai_like import OpenAILike
from llama_index.llms.llama_api import LlamaAPI



import pickle
import os
import gc
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SemanticSplitterNodeParser

import pickle
import os
import gc
import faiss
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.faiss import FaissVectorStore



/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Document Loading with Docling

In [36]:
document_paths = [
    "Regulations/CSRD.pdf",
    "Regulations/Celesia_CSRD.pdf"
]

# Output directories and files
output_dir = "markdown_outputs"
docs_save_path = os.path.join(output_dir, "processed_documents.pkl")

def load_documents_with_docling(force_recompute=False):
    """
    Load documents with Docling, only recomputing if forced or if files don't exist
    
    Args:
        force_recompute: If True, recompute even if files exist
        
    Returns:
        List of Document objects
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Check if we can use cached results
    all_files_exist = os.path.exists(docs_save_path)
    for source in document_paths:
        markdown_file = os.path.join(output_dir, f"{os.path.basename(source)}.md")
        if not os.path.exists(markdown_file):
            all_files_exist = False
            break
    
    # If all files exist and we're not forcing recompute, load from disk
    if all_files_exist and not force_recompute:
        print(f"📋 Loading previously processed documents from {docs_save_path}")
        with open(docs_save_path, "rb") as f:
            return pickle.load(f)
    
    # Otherwise, process documents
    print(f"🔄 {'Recomputing' if force_recompute else 'Computing'} document conversions...")
    
    # Force CPU for EasyOCR
    os.environ["EASYOCR_FORCE_CPU"] = "1"
    
    # Initialize processing
    load_dotenv()
    documents = []
    
    for source in document_paths:
        print(f"\n{'='*80}\nPROCESSING: {source}\n{'='*80}")
        
        try:
            # Clear memory before processing
            gc.collect()
            
            # Create a new converter
            converter = DocumentConverter()
            
            # Convert document
            result = converter.convert(source)
            
            # Get content as markdown
            markdown_content = result.document.export_to_markdown()
            
            # Save markdown
            output_filename = os.path.join(output_dir, f"{os.path.basename(source)}.md")
            with open(output_filename, "w", encoding="utf-8") as f:
                f.write(markdown_content)
            print(f"📄 Full markdown saved to: {output_filename}")
            
            # Show preview
            print(f"\n📄 Document Content Preview:")
            preview_length = 500
            print(f"{markdown_content[:preview_length]}...\n")
            print(f"📏 Total document length: {len(markdown_content)} characters")
            
            # Create document object
            doc = Document(
                text=markdown_content, 
                metadata={"source": source}
            )
            documents.append(doc)
            print(f"✅ Loaded document with Docling: {source}")
            
        except Exception as e:
            print(f"❌ Error loading document {source}: {e}")
        
        # Clear memory
        del converter
        gc.collect()
    
    # Save documents for later steps
    with open(docs_save_path, "wb") as f:
        pickle.dump(documents, f)
    print(f"\n📦 Saved {len(documents)} Document objects to: {docs_save_path}")
    
    return documents

# Example usage:
documents = load_documents_with_docling(force_recompute=True)

🔄 Recomputing document conversions...

PROCESSING: Regulations/CSRD.pdf
📄 Full markdown saved to: markdown_outputs/CSRD.pdf.md

📄 Document Content Preview:
## DIRECTIVE (EU) 2022/2464 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL

## of 14 December 2022

amending Regulation (EU) No 537/2014, Directive 2004/109/EC, Directive 2006/43/EC and Directive 2013/34/EU, as regards corporate sustainability reporting

## (Text with EEA relevance)

THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION,

Having regard to the Treaty on the Functioning of the European Union, and in particular Articles 50 and 114 thereof,

Having regard to the proposal from...

📏 Total document length: 268670 characters
✅ Loaded document with Docling: Regulations/CSRD.pdf

PROCESSING: Regulations/Celesia_CSRD.pdf
📄 Full markdown saved to: markdown_outputs/Celesia_CSRD.pdf.md

📄 Document Content Preview:
## The Ultimate Guide to the Corporate Sustainability Reporting Directive

For sustainability managers who wa

## Semantic chunking

In [37]:


# File paths
docs_path = "markdown_outputs/processed_documents.pkl"
nodes_save_path = "markdown_outputs/processed_nodes.pkl"
chunks_file = "all_chunks.txt"

def create_semantic_chunks(force_recompute=False):
    """
    Create semantic chunks from documents, only recomputing if forced or files don't exist
    
    Args:
        force_recompute: If True, recompute even if files exist
        
    Returns:
        List of node objects
    """
    # Check if we can use cached results
    if os.path.exists(nodes_save_path) and os.path.exists(chunks_file) and not force_recompute:
        print(f"📋 Loading previously processed chunks from {nodes_save_path}")
        with open(nodes_save_path, "rb") as f:
            return pickle.load(f)
    
    # Otherwise, create chunks
    print(f"🔄 {'Recomputing' if force_recompute else 'Computing'} semantic chunks...")
    
    # Load documents
    if not os.path.exists(docs_path):
        print("❌ No document data found. Please run document loading first.")
        return None
    
    with open(docs_path, "rb") as f:
        documents = pickle.load(f)
        
    print(f"📦 Loaded {len(documents)} documents for chunking")
    
    # Initialize embedding model
    embed_model = HuggingFaceEmbedding(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        device="cpu"
    )
    
    # Create semantic splitter
    text_parser = SemanticSplitterNodeParser(
        buffer_size=1, 
        breakpoint_percentile_threshold=80, 
        embed_model=embed_model
    )
    
    print("🧠 Using semantic chunking for text splitting")
    
    # Process all documents to get chunks
    print("⏳ Extracting semantic chunks...")
    all_nodes = text_parser.get_nodes_from_documents(documents)
    print(f"✂️ Split into {len(all_nodes)} chunks")
    
    # Save chunks to text file
    print(f"Saving {len(all_nodes)} chunks to {chunks_file}...")
    
    # Group nodes by source
    source_to_nodes = {}
    for node in all_nodes:
        source = node.metadata.get("source", "Unknown")
        if source not in source_to_nodes:
            source_to_nodes[source] = []
        source_to_nodes[source].append(node)
    
    # Write to file
    with open(chunks_file, "w", encoding="utf-8") as outfile:
        outfile.write("=" * 100 + "\n")
        outfile.write(f"ALL SEMANTIC CHUNKS\n")
        outfile.write(f"Total chunks: {len(all_nodes)}\n")
        outfile.write("=" * 100 + "\n\n")
        
        chunk_counter = 1
        
        # Process each source
        for source, source_nodes in source_to_nodes.items():
            outfile.write("\n" + "=" * 100 + "\n")
            outfile.write(f"SOURCE: {source}\n")
            outfile.write("=" * 100 + "\n\n")
            
            # Write each node for this source
            for node in source_nodes:
                text = node.text.strip()
                node_id = node.node_id if hasattr(node, 'node_id') else f"node_{chunk_counter}"
                
                # Write node header
                outfile.write("#" * 100 + "\n")
                outfile.write(f"CHUNK {chunk_counter} | ID: {node_id}\n")
                
                # If node has start/end position info, include it
                if hasattr(node, 'start_char_idx') and hasattr(node, 'end_char_idx'):
                    outfile.write(f"Character Range: {node.start_char_idx} - {node.end_char_idx} | ")
                
                outfile.write(f"Length: {len(text)} characters\n")
                outfile.write("-" * 100 + "\n\n")
                
                # Write the text
                outfile.write(text)
                
                # Add separator after node content
                outfile.write("\n\n" + "-" * 100 + "\n\n")
                
                chunk_counter += 1
    
    print(f"✅ Successfully wrote all chunks to: {chunks_file}")
    
    # Save nodes for vector store creation
    with open(nodes_save_path, "wb") as f:
        pickle.dump(all_nodes, f)
    print(f"📦 Saved nodes to: {nodes_save_path}")
    
    # Free up memory
    gc.collect()
    
    return all_nodes

# Example usage:
nodes = create_semantic_chunks(force_recompute=True)

🔄 Recomputing semantic chunks...
📦 Loaded 2 documents for chunking
🧠 Using semantic chunking for text splitting
⏳ Extracting semantic chunks...


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.83it/s]


✂️ Split into 254 chunks
Saving 254 chunks to all_chunks.txt...
✅ Successfully wrote all chunks to: all_chunks.txt
📦 Saved nodes to: markdown_outputs/processed_nodes.pkl


## Building Vector Store

In [38]:


# File paths
nodes_path = "markdown_outputs/processed_nodes.pkl"
vector_db_dir = "./faiss_semantic_chunking"

def build_vector_store(force_recompute=False):
    """
    Build vector store from nodes, only recomputing if forced or if store doesn't exist
    
    Args:
        force_recompute: If True, rebuild even if store exists
        
    Returns:
        VectorStoreIndex object
    """
    # Check if vector store already exists
    if os.path.exists(vector_db_dir) and os.path.isdir(vector_db_dir) and not force_recompute:
        try:
            print(f"📋 Loading existing vector store from {vector_db_dir}")
            embed_model = HuggingFaceEmbedding(
                model_name="sentence-transformers/all-MiniLM-L6-v2",
                device="cpu"
            )
            vector_store = FaissVectorStore.from_persist_dir(vector_db_dir)
            storage_context = StorageContext.from_defaults(
                vector_store=vector_store,
                persist_dir=vector_db_dir
            )
            index = VectorStoreIndex.from_storage_context(
                storage_context=storage_context,
                embed_model=embed_model
            )
            print("✅ Successfully loaded existing vector store")
            return index
        except Exception as e:
            print(f"⚠️ Error loading existing vector store: {e}")
            print("Will rebuild vector store...")
            force_recompute = True
    
    # If we need to build the vector store
    print(f"🔄 {'Rebuilding' if force_recompute else 'Building'} vector store...")
    
    # Load nodes
    if not os.path.exists(nodes_path):
        print("❌ No processed nodes found. Please run chunking first.")
        return None
    
    with open(nodes_path, "rb") as f:
        all_nodes = pickle.load(f)
    
    print(f"📦 Loaded {len(all_nodes)} nodes for vector store creation")
    
    # Build vector store
    print("⏳ Building vector index (this may take some time)...")
    
    # Initialize embedding model
    embed_model = HuggingFaceEmbedding(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        device="cpu"
    )
    
    # Create FAISS index
    d = 384  # Dimension of all-MiniLM-L6-v2 embeddings
    faiss_index = faiss.IndexFlatL2(d)
    vector_store = FaissVectorStore(faiss_index=faiss_index)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    
    # Create index with all nodes
    index = VectorStoreIndex(
        all_nodes,
        storage_context=storage_context,
        embed_model=embed_model
    )
    
    # Save to disk
    os.makedirs(vector_db_dir, exist_ok=True)
    storage_context.persist(persist_dir=vector_db_dir)
    
    print(f"🚀 FAISS Vector DB built and saved to {vector_db_dir}")
    print(f"✅ Vector store contains chunks from all processed documents")
    
    # Free up memory
    gc.collect()
    
    return index

# Example usage:
index = build_vector_store(force_recompute=True)

🔄 Rebuilding vector store...
📦 Loaded 254 nodes for vector store creation
⏳ Building vector index (this may take some time)...


Batches: 100%|██████████| 1/1 [00:00<00:00, 15.68it/s]


🚀 FAISS Vector DB built and saved to ./faiss_semantic_chunking
✅ Vector store contains chunks from all processed documents


## MAIN EXECUTION CELL (Run all with caching)


In [33]:
# Run the complete pipeline with smart caching
# Set force parameters to True to rebuild specific parts

# 1. Load documents with Docling
documents = load_documents_with_docling(force_recompute=False)  # When Python sees we're calling the load_documents_with_docling() function, It jumps to that function's definition

# 2. Create semantic chunks
nodes = create_semantic_chunks(force_recompute=False)

# 3. Build vector store
index = build_vector_store(force_recompute=False)

print("\n✅ PIPELINE EXECUTION COMPLETE")
print(f"Documents: {len(documents) if documents else 0}")
print(f"Chunks: {len(nodes) if nodes else 0}")
print(f"Vector store: {'Created successfully' if index else 'Failed'}")

🔄 Recomputing document conversions...

PROCESSING: Regulations/CSRD.pdf


KeyboardInterrupt: 